<a href="https://colab.research.google.com/github/Karen-Kwatia/lab-4-llm-decision-support/blob/main/Lab4_llm_decision_support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# API-key setup — DO NOT hard-code your key in this cell.

import os
# --- Google Colab (Secrets panel) ---
# TODO: set API_KEY using ONE of the methods above.
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")


# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


SECTION 1-TALKING TO AN LLM PROGRAMMATICALLY


PART 1.1 -YOUR FIRST API CALL

In [3]:
# TODO: Write a helper function you will reuse for the WHOLE lab:


def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
expected_answer=ask_llm("What is the name of the president of Ashesi University?")
print(expected_answer)

# TODO: Print response.usage as well — how many tokens did your call consume?
response_usage=  client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the name of the president of Ashesi University?"}],
)
print(response_usage)


The founder and president of Ashesi University is Patrick Awuah. He is a Ghanaian educator and entrepreneur who founded Ashesi University in 2002.
ChatCompletion(id='chatcmpl-5b79a972-d401-408a-bf1d-4bf0c9a419b1', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The President of Ashesi University is Patrick Awuah Jr.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1786728870, model='llama-3.3-70b-versatile', object='chat.completion', moderation=None, service_tier='on_demand', system_fingerprint='fp_ce7bc1685b', usage=CompletionUsage(completion_tokens=14, prompt_tokens=47, total_tokens=61, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.270746804, prompt_time=0.006200135, completion_time=0.083193274, total_time=0.089393409), usage_breakdown=None, x_groq={'id': 'req_01m00ndvypevdsqxghfkr5vew9', 'seed': 2087741220})


STUDENT REASONING
1. The system gives instructions to AI model before conversations start. It sets boundaries on what the model can do and how its personality should look like.
The user is the human operator that is going to use the model,the user submits questions and tasks to the model and expects results at the end.

2.A token is an atomic unit of the question presented to the AI model.API providers bill per token rather than per requests because of different request lengths and the computations it has to compute. For example, building by requests would mean the question;"Who is the president of Ashesi University?" and the question " Using Ashesi University as a reference point, write a 5000 research paper on the university, the president, staff members, community and impact. Include informations about the year of foundation, its mission and values, alumni and current students, age range of students, careers of alumni, founding partners and scholarships available" would have the same price. But this is not fair, the second question is more longer and has more tokens in its request than the first question and would use more computations compared to the first one. API providers bill per tokens rather than requests to differentiate between the amount of work done. Longer requests with more tokens have higher computational costs than shorter requests with small number of tokens.

PART 1.2 -TEMPERATURE :THE RANDOMNESS DIAL


In [4]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
#Asking the same question when temperature is 0.0
print("TEMPERATURE IS 0.0")

for i in range(5):
    answer_to_question=ask_llm("Suggest a name for a savings product for market traders in Accra.",temperature=0.0)
    print(answer_to_question)
print()

print("TEMPERATURE IS 1.2")
for i in range(5):
    answer_to_question=ask_llm("Suggest a name for a savings product for market traders in Accra.",temperature=1.2)
    print(answer_to_question)
print()
# TODO: Print all 10 answers, grouped by temperature.


TEMPERATURE IS 0.0
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Souce**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Souce" is a play on the word "source," implying a reliable and trustworthy savings product.
4. **Market Booster**: This name suggests that the savings product can help market traders boost their businesses and achieve their financial goals.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, implying a sense of community and mutual support among market traders.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian term that means "honest" or "transparent," which could help build trust with potential customers.
7. **Traders' Fund**: This name is straightforward and emphasizes

When the temperature was 0.0 the model was predictable and repetitive. Most of the names like "Makola Save","Sika Saver", appeared across most of the answers and when the temperature was 1.2, the responses were varied and different names were generated. The difference between the temperatures is that when the temperature is high, the model gives varying answers and is unpredictable comapre to when it is low.

I think the temperature appropriate for the loan decision is the support system is 0.0 or a temperature close to 0.0. This is because loan decisions should be predictable and consistent. Increasing the temperature introduces unpredicatbility and randomness which is not fair. An approval of a loan decision should be consistent and not random.

SECTION 2-The Dataset: Loan Application Letters


In [5]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


SECTION 3 -Prompt Engineering for the Decision Support System

Part 3.1 — Component 1: Summarization

In [6]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
print("NAIVE ATTEMPT")
SUMMARY_PROMPT_V1="Summarize this"
print(ask_llm(f"{SUMMARY_PROMPT_V1} {LETTERS['L002']}"))
print(ask_llm(f"{SUMMARY_PROMPT_V1} {LETTERS['L006']}"))
print()

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
print("PROPER TEMPLATE ATTEMPT")
SUMMARY_PROMPT_V2="You are an assistant to a microfinance loan officer and you have received all these applications for loans.You are to review these applications. In your review, be factual, neutral and do not add any invented details and summarize these applications in 3-4 sentences "
print(ask_llm(f"{SUMMARY_PROMPT_V2} {LETTERS['L002']}"))

print(ask_llm(f"{SUMMARY_PROMPT_V2} {LETTERS['L006']}"))
print()




#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
user_prompt_template ="Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
print(ask_llm(user_prompt_template.format(letter_text=LETTERS['L002']),system_prompt=SUMMARY_PROMPT_V2,temperature=0))
print()
print(ask_llm(user_prompt_template.format(letter_text=LETTERS['L006']),system_prompt=SUMMARY_PROMPT_V2,temperature=0))
print()

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

NAIVE ATTEMPT
Kwame Boateng, a commercial driver in Kumasi, is requesting an urgent loan of GHS 25,000 to repair his vehicle's engine and settle personal debts. He's experiencing a slow business period but expects things to improve after the festive season. He doesn't have collateral and is relying on his future earnings to repay the loan, asking for help as soon as possible.
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience in these ventures, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within one year, once his businesses are successful, and is offering his trustworthiness as assurance, as he has no collateral to provide.

PROPER TEMPLATE ATTEMPT
Kwame Boateng, a commercial driver from Kumasi, has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He mentions t

VI's output was vague and loses some details. For example in L001,Kwame Boaateng is seeking the loan to pay for his trotro engine. V1 does not state this but just said Kwame is seeking the loan to pay for his vehicle,without stating the specific type of vehicle that Kwame is going to fix the engine of. V2 corrects this by specifying the type of vehicle engine, Kwame wants to fix.
V1's output has no restriction on the length and therefore runs quite loosely but V2's output has a constraint and hence the effect of that length constraint is seen and used.
Also, in V1 the language is interpretive whiles V2 remains neutral.For example in V1(L006) it says "he claims to be business minded" which is drawing meaning from the application letter but V2(L006) remain neutral by not drawing any meaning from the application letter.

QUESTION 2
No invented details is essential in this application because the loan is given based on merits and the application letter of each applicant contains the merit that the applicant has.If the model adds details that do not exist, it affects the decisions of the loan officer and the loan might be given to someone who does not have enough resources to pay.The failure mode of this is called hallucination. Hallucinations occur when the AI model generates information that is false, misleading or fabricated and presents it as though it is correct. When this occurs, the loan office cannot detect that this is false information and the purpose of the summary fails and fairness and accuracy is not achieved.

PART 3.2-Component 2: Structured extraction (JSON)


In [16]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON

#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
import json
import pandas as pd

extract_prompt = """You are an experienced data extraction agent for a microfinance loan agency. You are to
read the letter below and extract the following details from the letter:
1. applicant_name (string) - the name of the applicant
2. amount_ghs (number) - the amount the applicant is requesting for
3. purpose (string) - what the applicant is going to use the money for
4. monthly_profit_ghs (number or null) - the applicant's monthly profit, if not stated use null
5. has_collateral_or_guarantor (boolean) - whether the applicant has a collateral or a guarantor (true if there is a collateral/guarantor and false otherwise)
6. repayment_months (number or null) - how many months it will take to repay, if the applicant stated it

RULES
1. Do not mention any field that is not stated in the letter
2. Do not make any inference or guesses
3. If a field is not stated in the letter, use null
4. Return ONLY a JSON object with EXACTLY these keys:
applicant_name (string), amount_ghs (number), purpose (string),
monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
repayment_months (number or null)

EXAMPLE:
Hello, my name is Dentaa. I am a twenty-two year old sales assistant at the Accra Mall. Currently, I am earning 1200 cedis every month.
I am applying for a 5000 cedis loan to help me get a proper phone and content creation equipment to help me start my content creation.
I am expecting to earn approximately 1000 cedis from content creation at the end of every month and I expect to pay within 12 months.
I have spoken to my aunt who has agreed to be my guarantor to help me get this loan. Thank you for your time.

EXPECTED OUTPUT:
{{
  "applicant_name": "Dentaa",
  "amount_ghs": 5000,
  "purpose": "To start content creation",
  "monthly_profit_ghs": 1200,
  "has_collateral_or_guarantor": true,
  "repayment_months": 12
}}

Now extract the fields from this letter:

{letter_text}
"""



# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

def extract_fields(letter_text,temperature=0.0):
  extracted_output=ask_llm(extract_prompt.format(letter_text=letter_text),temperature=temperature)

  clean_output = extracted_output.strip().strip("`").replace("json","",1).strip()

  try:
    return json.loads(clean_output)
  except:
    print("WARNING-Failure occured")
    return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
collect_results=[]
for letter in LETTERS:
  collect_results.append(extract_fields(LETTERS[letter]))
df =pd.DataFrame(collect_results)
df




,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,Akosua Mensah,8000,to buy a deep freezer and expand into frozen f...,900.0,True,20.0
1,Kwame Boateng,25000,to repair my trotro engine and settle some per...,NaN,False,NaN
2,Efua Darko,15000,to purchase two industrial sewing machines and...,2800.0,True,15.0
3,Yaw Owusu,12000,for feed and 500 new layers for my poultry farm,1500.0,True,18.0
4,Adenta Women's Weaving Cooperative,30000,to buy a bulk order of yarn directly from the ...,NaN,True,16.0
5,Kofi,50000,"to start a car washing business, a provision s...",NaN,False,12.0


STUDENT REASONING
 1. Why must the few-shot example NOT come from the six letters you are processing? 2. Why "use null, do not guess" — what did the model do without that instruction? 3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?

 1. The few shot shot example must not come from the six letters because of information leaking. If it comes from the six letters we are processing the model will ot learn but rather generalize based on the training questions it was given. To get a good evaluation and training accuracy, a new prompt must be used that is different from the test set to ensure that a good testing accuracy is achieved. Also by creating my own question, I get to extract my own answers and use them as ground truth instead of depending on the LETTERS that were created for me.

 2. The model avoids hallucination by using null and not guessing. If something was not explicitly stated in the application letter, the model is not allowed to guess but rather use null indicating that there is no information on that part. This affects the model's accuracy because it does not guess or makes inference but rather simply extracts information based on what letter it is given.In the results above, NaN was used for the Kwame Boateng's application letter where monthly_profit_ghs and repayment_months were not explicitly and this follows the instructions exactly.

 3.temperature=0.0 is right for extraction but arguably not for creative tasks , because in extraction there is just one answer which is predictable and there is no need to introduce variety and randomness. In the extraction of details from a letter, we would want the model to extract specific identities which is predictable and does not need variety to ensure that the model is consistent. In creative tasks we need variety and setting the temperature to 0 prevents that variety and keeps on just generating the same outputs which does not help in creative artistry.


Part 3.3 — Component 3: The decision-support brief

In [9]:

# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
BRIEF_PROMPT = """You are an assistant to a loan officer working in a
microfinance company. You are to give a decision-support brief that helps
the loan officer to make a decision. Base every point strictly on the letter
and extracted data provided below and never invent or assume information.

These are your rules:
1. You are to give the strengths of the letter in bullet points.
2. You are to list the risks/red flags in the letter also in bullet points.
3. You are also to note down the missing information the officer needs to ask
   from the applicant.
4. You are to suggest ONE next step, chosen only from: "invite for interview",
   "request documents", or "flag for senior review". Do not suggest anything else.
5. You should never give a final decision (e.g. never say "approve" or "reject").
6. All final decisions are made by the human (loan officer)."""

#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

def build_brief_user_prompt(letter_text,extracted):
  return f"""LETTER
{letter_text}

EXTRACTED DATA
{json.dumps(extracted,indent=2)}

Generate the decision support brief following the brief prompt instructions

"""

#
# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
def generate_brief(letter_id,temperature =0.0):
  letter_text=LETTERS[letter_id]
  extracted=extract_fields(letter_text)
  if extracted is None:
    extracted={}
  user_prompt = build_brief_user_prompt(letter_text,extracted)

  brief = ask_llm(
      user_prompt,
      system_prompt=BRIEF_PROMPT,
      temperature=temperature,
  )
  return brief

briefs={}
for letter_id in LETTERS:
  briefs[letter_id] = generate_brief(letter_id)
for letter_id in ["L001", "L002", "L006"]:
  print(f"BRIEF FOR {letter_id}:")
  print(briefs[letter_id])
  print()
print(briefs["L003"])


BRIEF FOR L001:
**Decision Support Brief**

**Strengths:**
* The applicant has 12 years of experience in selling provisions, indicating a stable business.
* The applicant has a consistent profit of GHS 900 per month, demonstrating a viable income stream.
* The applicant has saved GHS 2,500 with the susu scheme over two years, showing a ability to save and commit to regular payments.
* The applicant has a guarantor, a teacher, which may provide an additional layer of security.
* The applicant has a clear plan for using the loan, to buy a deep freezer and expand into frozen foods.

**Risks/Red Flags:**
* The applicant's proposed monthly repayment of GHS 450 is approximately 50% of their monthly profit, which may be a high repayment burden.
* There is no information provided about the applicant's current debt obligations or credit history.
* The guarantor's financial capabilities and stability are not assessed or provided.

**Missing Information:**
* Current debt obligations or credit his

STUDENT REASONING

1. The system correctly identified the strengths of L003 and the weakness of L006.In L003 the applicant has a registered business which is a solid source of revenue, has records of past sales , a collateral and a repayemnt plan, but in L006 the applicant made statements about his energy and enthusiasm with no stable revenue stream. The system correctly identified that L003 is a more advantageous applicant because of her stable source of revenue, with a sale record and intentions.
The system also correctly identified L006 redflags as the applicant has no experience in the businesses he plans on using the loan for. The applicany also has no guarantor or collateral and his repayment plan depends on whether his business is booming or not.
The differences in both applicants was identified but however both applicant got the same next step which indicates a weakness in the system.


2.We forbade the model from making decisions whether to approve or reject because the model can hallucinate. If the model makes the final decision and the loan officer uses that judgement, real consequences might occur since loan approvals and rejections are affect the lives of human beings and the loan officer is held accountable since the AI agent cannot face accountability and legal action.

SECTION 3.4 -COMMITING PROMPT TEMPLATES


hash : 45625885762b6e93e27ba3f00416a40685a6bd07

SECTION 4-Evaluation: Quality, Reliability, Appropriateness

Part 4.1 — Extraction accuracy against gold labels

In [11]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
          "has_collateral_or_guarantor", "repayment_months"]

gold_letters = ["L001", "L003", "L006"]

results_table = {f: [] for f in fields}

for letter_id in gold_letters:
    extracted = extract_fields(LETTERS[letter_id])
    gold = GOLD[letter_id]
    for f in fields:
        ev = extracted.get(f) if extracted else None
        gv = gold.get(f)
        if f == "applicant_name":
            match = str(ev).strip().lower() == str(gv).strip().lower()
        elif f == "purpose":
            match = True  # free text — judged loosely, not exact string match
        else:
            match = ev == gv
        results_table[f].append("✓" if match else "✗")

comparison_df = pd.DataFrame(results_table, index=gold_letters).T
comparison_df["accuracy"] = (comparison_df[gold_letters] == "✓").mean(axis=1)
comparison_df






,L001,L003,L006,accuracy
applicant_name,✓,✓,✓,1.0
amount_ghs,✓,✓,✓,1.0
purpose,✓,✓,✓,1.0
monthly_profit_ghs,✓,✓,✓,1.0
has_collateral_or_guarantor,✓,✓,✓,1.0
repayment_months,✓,✓,✓,1.0


SECTION 4.2-Reliability: is the system consistent?

In [19]:

# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach:
#json.dumps(result, sort_keys=True)
#   and count unique strings.

results_for_temperature0 = [extract_fields(LETTERS["L004"],temperature=0.0) for i in range(5)]
results_for_temperature1 = [extract_fields(LETTERS["L004"],temperature=1.2) for i  in range(5)]

def reliability_report(results,label):
  valid = [r for r in results if r is not None]
  serialized = [json.dumps(r,sort_keys=True) for r in valid]
  unique = set(serialized)

  print(f"TEMPERATURE : {label}")
  print(f"Valid JSON : {len(valid)}/5")
  print(f"Unique outputs among valid runs : {len(unique)}")
  for i,r in enumerate(results):
    print(f"Run {i +1} : {r}")

  print()
reliability_report(results_for_temperature0,"0.0")
reliability_report(results_for_temperature1,"1.2")


TEMPERATURE : 0.0
Valid JSON : 5/5
Unique outputs among valid runs : 1
Run 1 : {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 2 : {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 3 : {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 4 : {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 5 : {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed

Part 4.3 — Hallucination probing

In [23]:

# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
print("TEST 1")
test1_prompt = f"Using this loan application, what is the applicant's credit score ? \n\n {LETTERS["L002"]}"
print(ask_llm(test1_prompt))
print()
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
print("TEST 2")
test2_prompt = """The Ghana Meteorological Agency have released a press with
the weather details. Tomorrow, the temperature would be around 24.4 degrees witha humidity
of about 86.9 % between times 12 am and 6am. We are expecting a mixture of
periodic clouds."""
print(ask_llm(extract_prompt.format(letter_text=test2_prompt),temperature=0))

print()
# TODO: Record the outputs verbatim below and label each PASS or FAIL.
##TEST 1 :PASS
##TEST 2 :PASS on content
#TEST 1
"""The applicant's credit score is not mentioned in the loan application. The application only provides information about the applicant's identity, occupation, loan amount needed, and intended use of the loan, but it does not include any details about their credit history or credit score.

TEST 2
Since the letter does not mention any details related to a loan application, the output will be null for all fields. However, to follow the exact format, here is the output:

{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}"""

TEST 1
The applicant's credit score is not mentioned in the loan application. The application only provides information about the applicant's identity, occupation, loan amount needed, and intended use of the loan, but it does not include any details about their credit history or credit score.

TEST 2
Since the letter does not mention any details related to a loan application, the output will be null for all fields. However, to follow the exact format, here is the output:

{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}



STUDENT REASONING

1.When I compared my results against the Gold labels for L001,L003 and L006 , the model matched on all the extracted features correctly that is applicant_name,amount_gh,monthly_profit_ghs,has_collateral_or_guarantor and repayment_months for all three and gave a 100% accuracy.The hardest field for the model was purpose since it is not a fixed value and varies.

2. The reliability experiment at both temperatures 0 and 1 produced 5/5 valid JSON outputs and all runs were identical in each case on L004. This suggests that the schema wuth the few shot examples were enough constraints to keep the model consistent even when the temperature was supposed to introduce randomness.

3. The system did not hallucinate under probing. In test 1 it correctly passed by saying the credit score was not mentioned in the application and did not incent one. In test 2 , it returned null for every factor it was to extract , which supports the instruction we gave it by saying it should use null and not guess. But it returned a sentence of explanation which breaks the JSON format of the output, and this can be fixed by adding a strict return only JSON constraint.

Part 4.4 — Appropriateness: should this system exist?

1.If the bank automated decisions using my system, people who run successful business but are not literates would be unfairly harmed. This is because my system summarizes and extracts information from what is written in the letter and nothing else, so if an illiterate applies for a loan with a letter that is not easily understandable by my model,it does not consider it a good application and creates bias towards some people who might not have good businesses but are literate.

2.Sending letters to a third party API abroad means sending personal and financial information to another country which might serve as data to be trained another model without the applicant's consent.
Before deploying I would check whether it is under compliance with the protection data laws of Ghana,and whether the applicants have given consent for their data to be used.

3. Two concrete safeguards I would build around this system are;

  a. Mandatory human review to ensure that the system is always acting as a helper to the loan officer and not acting like the loan officer by approving and rejecting applicants.

  b.consistent re-evaluation against the letters to ensure that poor performance of the model is caught and fixed to imrove accuracy.